# 00 — Data Time Fix

The robot crashed mid-run and restarted several times, producing multiple sub-folders per run with overlapping timestamps.

**Strategy:**
- Take all data from every sub-folder in full.
- For each subsequent segment, shift its timestamps so it starts `BUFFER_SECONDS` after the previous segment ended.
- Insert a single NaN marker row (at `last_t + NAN_OFFSET`) between segments so downstream code can detect the boundary.
- Keep the `%`-comment header from the first sub-folder only.
- Write merged files directly into `data/raw/Farm/RunX/`.

**It is not part of the pipeline** — only run this notebook to fix overlapping-timestamp issues from robot crashes.

In [21]:
from pathlib import Path
import numpy as np

In [22]:
# ── Configuration ─────────────────────────────────────────────────────────────
DATA_ROOT      = Path("../data/raw/Farm")
SENSOR_FILES   = [
    "log_t0_acc_1.txt",
    "log_t0_encoder_velocity.txt",
    "log_t0_gyro_1.txt",
    "log_t0_pose.txt",
]



In [23]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def read_sensor_file(filepath):
    """Return (header_lines, data) where data is an (N, C) float64 array."""
    header, rows = [], []
    with open(filepath) as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith("%"):
                header.append(line)
            elif line.strip():
                rows.append([float(x) for x in line.split()])
    return header, np.array(rows, dtype=float)


def write_sensor_file(filepath, header_lines, data):
    """Write header comments followed by space-separated data rows."""
    with open(filepath, "w") as f:
        for h in header_lines:
            f.write(h + "\n")
        for row in data:
            parts = []
            for v in row:
                if np.isnan(v):
                    parts.append("NaN")
                else:
                    parts.append(f"{v:.4f}")
            f.write(" ".join(parts) + "\n")


def merge_segments(segments):
    """
    Concatenate segments with adjusted timestamps.

    Each segment after the first is shifted so it starts exactly
    after the previous segment's last timestamp.
    """
    n_cols = segments[0].shape[1]
    parts = [segments[0]]
    current_end = segments[0][-1, 0]

    for seg in segments[1:]:
        seg_start = seg[0, 0]
        # Shift so this segment starts right after the previous end
        shift = current_end - seg_start

        # Shifted segment
        shifted = seg.copy()
        shifted[:, 0] += shift
        parts.append(shifted)
        current_end = shifted[-1, 0]

    return np.vstack(parts)


def process_run(run_dir, sensor_files):
    subfolders = sorted([d for d in run_dir.iterdir() if d.is_dir()])
    if not subfolders:
        print(f"  [SKIP] No sub-folders found in {run_dir.name}")
        return

    print(f"  Sub-folders ({len(subfolders)}): {[d.name for d in subfolders]}")

    for sensor_file in sensor_files:
        print(f"\n  {sensor_file}")
        segments, header_lines = [], None

        for folder in subfolders:
            fp = folder / sensor_file
            if not fp.exists():
                print(f"    WARNING: {fp.name} not found in {folder.name} — skipping")
                continue
            header, data = read_sensor_file(fp)
            if header_lines is None:
                header_lines = header
            segments.append(data)
            print(f"    {folder.name}: {len(data):>6} rows  "
                  f"t=[{data[0,0]:.4f} … {data[-1,0]:.4f}]")

        if not segments:
            print(f"    [SKIP] No data found for {sensor_file}")
            continue

        merged = merge_segments(segments)
        out_path = run_dir / sensor_file
        write_sensor_file(out_path, header_lines, merged)

        nan_count = int(np.sum(np.isnan(merged[:, 1])))
        print(f"    → saved: {len(merged)} rows, {nan_count} NaN boundary markers, "
              f"t=[{merged[0,0]:.4f} … {merged[-1,0]:.4f}]")

In [24]:
# ── Process all runs ───────────────────────────────────────────────────────────
runs = sorted([d for d in DATA_ROOT.iterdir() if d.is_dir() and d.name.startswith("Run")])
print(f"Runs found: {[r.name for r in runs]}\n")

for run_dir in runs:
    print("=" * 60)
    print(f"Processing {run_dir.name} …")
    process_run(run_dir, SENSOR_FILES)

print("\n" + "=" * 60)
print("Done.")

Runs found: ['Run1', 'Run2']

Processing Run1 …
  Sub-folders (3): ['log_20260421_114126.191', 'log_20260421_114126.852', 'log_20260421_114127.477']

  log_t0_acc_1.txt
    log_20260421_114126.191:   5807 rows  t=[1776764504.9092 … 1776764574.5822]
    log_20260421_114126.852:  13524 rows  t=[1776764505.4699 … 1776764667.8892]
    log_20260421_114127.477:  24663 rows  t=[1776764506.2748 … 1776764802.2628]
    → saved: 43994 rows, 0 NaN boundary markers, t=[1776764504.9092 … 1776765032.9895]

  log_t0_encoder_velocity.txt
    log_20260421_114126.191:   5727 rows  t=[1776764504.9063 … 1776764573.6187]
    log_20260421_114126.852:  13101 rows  t=[1776764505.4685 … 1776764662.8009]
    log_20260421_114127.477:  24666 rows  t=[1776764506.2731 … 1776764802.2953]
    → saved: 43494 rows, 0 NaN boundary markers, t=[1776764504.9063 … 1776765026.9733]

  log_t0_gyro_1.txt
    log_20260421_114126.191:   7863 rows  t=[1776764504.9090 … 1776764599.2812]
    log_20260421_114126.852:  14443 rows  t=[

In [25]:
# ── Verification ──────────────────────────────────────────────────────────────
print("Merged file summary\n" + "─" * 60)

for run_dir in runs:
    print(f"\n{run_dir.name}:")
    for sensor_file in SENSOR_FILES:
        out_path = run_dir / sensor_file
        if not out_path.exists():
            print(f"  {sensor_file}: NOT FOUND")
            continue
        _, data = read_sensor_file(out_path)
        nan_mask = np.isnan(data[:, 1])
        boundary_times = data[nan_mask, 0]
        print(f"  {sensor_file}:")
        print(f"    rows={len(data)}, boundaries={nan_mask.sum()}, "
              f"t_start={data[0,0]:.4f}, t_end={data[-1,0]:.4f}")
        for b_t in boundary_times:
            idx = int(np.where(data[:, 0] == b_t)[0][0])
            t_before = data[idx - 1, 0] if idx > 0 else float("nan")
            t_after  = data[idx + 1, 0] if idx < len(data) - 1 else float("nan")
            print(f"    boundary @ t={b_t:.4f}  "
                  f"prev={t_before:.4f}  next={t_after:.4f}  "
                  f"gap={t_after - t_before:.3f}s")

Merged file summary
────────────────────────────────────────────────────────────

Run1:
  log_t0_acc_1.txt:
    rows=43994, boundaries=0, t_start=1776764504.9092, t_end=1776765032.9895
  log_t0_encoder_velocity.txt:
    rows=43494, boundaries=0, t_start=1776764504.9063, t_end=1776765026.9733
  log_t0_gyro_1.txt:
    rows=47025, boundaries=0, t_start=1776764504.9090, t_end=1776765069.3770
  log_t0_pose.txt:
    rows=78552, boundaries=0, t_start=1776764504.9060, t_end=1776765054.9917

Run2:
  log_t0_acc_1.txt:
    rows=76431, boundaries=0, t_start=1776764809.3880, t_end=1776765726.7657
  log_t0_encoder_velocity.txt:
    rows=75921, boundaries=0, t_start=1776764809.3862, t_end=1776765720.6085
  log_t0_gyro_1.txt:
    rows=75120, boundaries=0, t_start=1776764809.3876, t_end=1776765711.0160
  log_t0_pose.txt:
    rows=128903, boundaries=0, t_start=1776764809.3847, t_end=1776765711.9996
